# KAT-VAD v5.1 — tile-scored MQR: grids, R1c / R3c / R6c (s2024), pilot read-out

Rules: `core/docs/v5/PREREG_V5.md` **§3** (signed 2026-10-29: fixed λ 0.5 / K 4 / β 0.01, **one seed s2024**, R5c not
run, PASS rule (i)–(iii)). Plan `.project/plans/katvad-v5-tile-mqr.md`. The read-out is labelled **PRELIMINARY
(n = 1)** and adopts nothing; PILOT-PASS → 3-seed confirmation (§3.4), PILOT-FAIL → written up as a negative.

| step | what | runs in parallel | GPU? |
|---|---|---|---|
| 1 | centred **4×7×7** grids: Nexar train **s8**, Nexar val **s3**, DoTA-CAP **s3** (5 CAP groups in order) + `v5_head.npz` | the 3 caches at once | yes |
| 2 | **train** R1c, R3c, R6c (A2's recipe + `v5.*`, 1,749 steps) → J10 + S1-G4 provenance + finite losses | the 3 runs at once | yes |
| 3 | reads: `nexar_val` (guardrail), DoTA-CAP-dev protocol B, the same **time-reversed**, maps dump (R3c, R6c) | `MAX_GPU_JOBS` at once | yes |
| 4 | maps read (pointing, mass-in-box, STAUC vs uniform / centre / change) + **pilot read-out** → paste `REPORTS/v51_pilot_readout.md` to Claude | — | no |

**Parallelism.** GPU steps start independent jobs as separate processes on the one GPU (`MAX_GPU_JOBS`, default 3;
V2-S is 22 M params and every job here is decode- or I/O-bound, so they overlap well on an A100). CPU threads are split
between jobs (`OMP_NUM_THREADS`). If a job dies of CUDA OOM set `MAX_GPU_JOBS = 1` and re-run the cell: finished jobs
are skipped. Concurrency changes no rule and no number (§3.6).

**Grids are written to VM disk** and used from there by steps 2–3 (no 31 GB read back from Drive); a background
mirror copies every finished file to Drive every few minutes, so a disconnect loses at most that window. After a
reconnect step 1 restores Drive's partial copy and the tools skip finished videos.

**Cost (§3.5, estimate).** Grids ≈ 31 GB on Drive (12.3 + 12.9 + 5.6). Sequential ≈ 7–10 A100-h; with the parallel
steps expect ≈ 4–6 h wall (step 1 ≈ 2–3 h, bound by the CAP tar stream; step 2 ≈ 1 h; step 3 ≈ 0.5–1 h).
Runtime: **A100, High-RAM**, `/content` ≥ 80 GB free.

**Drive:** code `Thesis-V5/kat-vad` (re-upload this branch, `v5-probe`); writes only to
`Thesis-V5/cache/video_grid/<V2-S>/*_grid4x7/` and `Thesis-V5/outputs/v5/v5_tile/`; `Thesis/` + `Thesis-V2/` are
read-only.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (Nexar, DoTA, MMAU/CAP), cache/clip, ckpts
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: v2 caches / runs / reads, DoTA-CAP alignment
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # code + everything v5.1 writes
os.environ['PROJECT_ROOT']       = DRIVE_V5
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V5}/outputs'

os.environ['REPO'] = f'{DRIVE_V5}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5-probe branch there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_TILE_ROWS'), 'this upload predates v5.1 -- re-upload kat-vad/ from v5-probe'
assert (REPO / 'core/tools/v5_tile_read.py').is_file(), 'no core/tools/v5_tile_read.py in this upload'
prereg = (REPO / 'core/docs/v5/PREREG_V5.md').read_text(encoding='utf-8')
assert '**Status: SIGNED 2026-10-29' in prereg.split('## §3 v5.1', 1)[1], 'PREREG_V5 §3 is not signed in this upload'

MAX_GPU_JOBS = 3      # concurrent GPU processes (steps 1-3); 1 = sequential
MIRROR_EVERY = 300    # seconds between grid mirrors VM -> Drive during step 1

ENC      = constants.V5_ENCODER                                   # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW                    # 'window'
DATASET  = constants.NEXAR_DATASET
ROWS     = list(constants.V5_TILE_PILOT_ROWS)                     # R1c, R3c, R6c
CONTROL  = ROWS[0]                                                # R1c
MAP_ROWS = [r for r in ROWS if constants.V5_TILE_ROWS[r]['readout'] != constants.V5_READOUT_MEAN]
SEEDS    = [constants.V5_TILE_SEED]                               # s2024 (§3 T7)
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
TAG      = constants.V5_TILE_GRID_TAG                             # grid4x7
S_TRAIN, S_VAL, S_DOTA = (constants.V5_TILE_NEXAR_TRAIN_STRIDE, constants.V5_TILE_NEXAR_VAL_STRIDE,
                          constants.V5_TILE_DOTA_STRIDE)          # 8, 3, 3

V2_CACHE = Path(DRIVE_V2) / 'cache'
V2_OUT   = Path(DRIVE_V2) / 'outputs'
RAW      = constants.DATA_ROOT / DATASET / 'raw'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1 = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'   # G-c reference (Nexar)
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
CAP_RAW  = constants.DATA_ROOT / 'MMAU' / 'raw'
DCAP     = V2_OUT / 'REPORTS' / 'dota_cap'
ALIGN, FINAL = DCAP / 'dota_cap_alignment.json', DCAP / 'dota_cap_readout.json'
CAP_SQUASH = next((p for p in (constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash',
                               V2_CACHE / 'video' / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash') if p.exists()),
                  None)                                                                 # G-c reference (DoTA-CAP)
CAP_GROUPS = ['1-10', '11', '12-42', '43', '44-62']

GRIDS    = Path(DRIVE_V5) / 'cache' / constants.V5_GRID_CACHE_DIRNAME / ENC
D_GRID   = {'train': GRIDS / f'{DATASET}_s{S_TRAIN}_{TAG}', 'val': GRIDS / f'{DATASET}_s{S_VAL}_{TAG}',
            'dota': GRIDS / f'{constants.DOTA_CAP_DATASET}_s{S_DOTA}_{TAG}'}
ROOT     = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_TILE_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/v51')
L_GRID   = {k: LOCAL / 'grids' / v.name for k, v in D_GRID.items()}
L_REP    = LOCAL / 'reports'          # every tool writes here; mirrored to REPORTS
LOGS     = LOCAL / 'logs'
for p in (REPORTS, L_REP, LOGS, *D_GRID.values(), *L_GRID.values()):
    p.mkdir(parents=True, exist_ok=True)
assert constants.OUTPUT_ROOT == Path(DRIVE_V5) / 'outputs', constants.OUTPUT_ROOT

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']                     # 1,749 = 33 x 53 (v2 §29)
for name, path in (('Nexar raw', RAW), ('census', CENSUS), ('Nexar_window', DATA), ('clip s1', CLIP_S1),
                   ('V2-S Nexar s1', VIDEO_S1), ('A2 bake', A2_BAKE), ('KNN', KNN), ('v2 A2 runs', A2_RUNS),
                   ('DoTA s1', DOTA_S1), ('DoTA-CAP-dev A2 bake', DOTA_A2), ('DoTA labels', DOTA / 'labels_s8'),
                   ('CAP chunks', CAP_RAW / 'CAP-DATA_chunks'), ('CAP annotation', CAP_RAW / 'cap_text_annotations.xls'),
                   ('alignment', ALIGN), ('final ids', FINAL), ('V2-S DoTA-CAP s1', CAP_SQUASH)):
    assert path is not None and path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}   OK')
print(f'rows {ROWS} | seeds {SEEDS} | {EPOCHS} epochs = {STEPS} steps | GPU jobs {MAX_GPU_JOBS}')
print('grids ->', GRIDS, '\nruns + REPORTS ->', ROOT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd pandas
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 1-3 need one'
nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import threading
import time

import numpy as np
import torch

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
CPUS = os.cpu_count() or 8
PART = constants.CACHE_PART_SUFFIX
assert shutil.disk_usage('/content').free / 1e9 >= 80, 'need >= 80 GB free on /content (grids 31 GB + CAP stream)'


def run(cmd, log=None):
    '''One CPU / sequential command, output streamed here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def last_line(path):
    try:
        lines = [l for l in path.read_text(errors='replace').splitlines() if l.strip()]
    except FileNotFoundError:
        return '(no output yet)'
    return lines[-1][-160:] if lines else '(no output yet)'


class Job:
    '''Commands run in order in one process slot; ``after`` runs in this kernel once all succeeded.'''

    def __init__(self, name, cmds, after=None):
        self.name, self.after = name, after
        self.cmds = [[str(c) for c in cmd] for cmd in cmds]
        self.log = LOGS / f'{name}.log'


def run_parallel(jobs, max_jobs=MAX_GPU_JOBS, every=120):
    '''Run independent jobs ``max_jobs`` at a time, one GPU. A failure does not stop the others; the cell raises
    after all have ended, with each failed job's log tail.'''
    if not jobs:
        print('nothing to run')
        return
    slots = max(1, min(max_jobs, len(jobs)))
    env = dict(ENV, OMP_NUM_THREADS=str(max(1, CPUS // slots)), MKL_NUM_THREADS=str(max(1, CPUS // slots)))
    pending, running, failed, t0, shown = list(jobs), {}, [], time.time(), time.time()

    def launch(job, i):
        fh = open(job.log, 'a')
        fh.write(f'\n$ {" ".join(job.cmds[i])}\n')
        fh.flush()
        return subprocess.Popen(job.cmds[i], cwd=str(REPO), env=env, stdout=fh, stderr=subprocess.STDOUT), fh

    def stamp():
        return f'[{(time.time() - t0) / 60:6.1f} min]'

    while pending or running:
        while pending and len(running) < slots:
            job = pending.pop(0)
            job.t0 = time.time()
            running[job.name] = (job, 0, *launch(job, 0))
            print(f'{stamp()} start {job.name}  (log {job.log})')
        time.sleep(10)
        for name, (job, i, proc, fh) in list(running.items()):
            rc = proc.poll()
            if rc is None:
                continue
            fh.close()
            del running[name]
            if rc:
                failed.append(job)
                print(f'{stamp()} FAILED {name} (exit {rc})')
            elif i + 1 < len(job.cmds):
                running[name] = (job, i + 1, *launch(job, i + 1))
            else:
                try:
                    if job.after:
                        job.after()
                    print(f'{stamp()} done {name} in {(time.time() - job.t0) / 60:.1f} min')
                except Exception as exc:  # a failed publish / check is this job's failure, not the others'
                    failed.append(job)
                    print(f'{stamp()} FAILED {name} after the run: {exc!r}')
        if running and time.time() - shown > every:
            shown = time.time()
            for name, (job, *_rest) in running.items():
                print(f'           {name}: {last_line(job.log)}')
    if failed:
        for job in failed:
            tail = job.log.read_text(errors='replace').splitlines()[-30:]
            print('=' * 30, f'\n{job.name} -- paste to Claude:\n' + '\n'.join(tail))
        raise RuntimeError(f'failed: {[j.name for j in failed]}')


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file() and not p.name.endswith(PART)]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.partial')
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def sync_dir(src, dst, restore=False):
    '''Copy every finished file of ``src`` missing (or, for json / npz, changed) in ``dst``; in-flight writes skipped.'''
    n = 0
    for f in sorted(src.glob('*')):
        if not f.is_file() or f.name.endswith(PART) or f.name.endswith('.partial'):
            continue
        g = dst / f.name
        small = f.suffix in ('.json', '.npz', '.md')
        if g.exists() and (g.stat().st_size == f.stat().st_size) and not (small and not restore):
            continue
        if restore:
            dst.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, g)
        else:
            publish(f, g)
        n += 1
    return n


class Mirror(threading.Thread):
    '''Step 1: copy finished grid files + reports VM -> Drive every ``MIRROR_EVERY`` s, and once more at stop.'''

    def __init__(self, pairs):
        super().__init__(daemon=True)
        self.pairs, self.halt, self.copied = pairs, threading.Event(), 0

    def run(self):
        while not self.halt.wait(MIRROR_EVERY):
            self.sync()

    def sync(self):
        for src, dst in self.pairs:
            try:
                self.copied += sync_dir(src, dst)
            except OSError as exc:     # a Drive hiccup: the next round retries
                print('mirror:', exc)

    def stop(self):
        self.halt.set()
        self.join()
        self.sync()
        print(f'mirror: {self.copied} files copied to Drive')


def ckpt_step(path):
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def show(name):
    p = L_REP / f'{name}.md'
    print(p.read_text() if p.exists() else f'(no {p.name})')


sync_dir(REPORTS, L_REP, restore=True)     # earlier sessions' reports: every skip below reads them locally
print('reports restored:', sorted(p.name for p in L_REP.glob('*.json')))

## 1. Centred 4×7×7 grids — three caches in parallel (GPU)

* **Nexar train** positives at **s8** and **Nexar val** at **s3** (`v5_stage1 extract --tile`): S1-G1 length gate;
  G-c cos vs v2's causal `Nexar_s1_squash` row `t + 21` printed (§2.4 B9).
* **DoTA-CAP** (all 1,129 clips, the 5 CAP groups one after the other in one job) at **s3**
  (`v5_stage0 extract --tile`): G-c per clip, cos ≥ 0.9999 vs `DoTA_CAP_s1_squash` row `t + 7`; a failing clip
  stores nothing. **Pass = every CAP-dev clip stored** (the pilot reads all of them).
* `v5_head.npz` (V2-S `fc_norm` + Nexar A2's `c`, `m_u`, `σ_u`) into the train cache.

A cache whose gate already PASSed on Drive is not re-extracted; its files are copied to VM disk only when a later step
needs them.

In [ ]:
from core.data.v2_splits import load_split
from core.data.windows import load_windows

census = json.loads(CENSUS.read_text())
TRAIN_IDS = sorted({w.source for w in load_windows(DATA).values()})
VAL_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_VAL))
CAP_IDS = sorted(load_split(CAP_DEV))
assert all(census['videos'][v]['label'] == 1 for v in TRAIN_IDS), 'a window source is not a positive'


def nexar_gate(stride, ids):
    p = L_REP / f'g1_extract_tile_s{stride}.json'
    if not p.exists():
        return False
    r = json.loads(p.read_text())
    return r['pass'] and r['videos'] == len(ids) and r['ok'] == len(ids)


def dota_summary():
    '''Combine the 5 group reports: per-reason counts; pass iff every CAP-dev clip is stored (§3.2).'''
    report = {}
    for g in CAP_GROUPS:
        p = L_REP / f'extract_tile_dota_{g}.json'
        if p.exists():
            report |= json.loads(p.read_text())
    ok = {v for v, e in report.items() if e.get('reason') == 'ok'}
    failed = {v: e.get('reason') for v, e in report.items() if e.get('reason') != 'ok'}
    missing_dev = sorted(set(CAP_IDS) - ok)
    cos = [e['g0_min_cos'] for e in report.values() if e.get('g0_min_cos') is not None]
    s = {'rule': 'PREREG_V5 §3.2 G-c (DoTA-CAP, centred 4x7x7, s3)', 'clips_reported': len(report),
         'ok': len(ok), 'failed': failed, 'cap_dev_missing': missing_dev,
         'g_c_min_cos': min(cos, default=None), 'pass': len(report) >= 1129 and not missing_dev}
    (L_REP / 'g_c_dota_tile.json').write_text(json.dumps(s, indent=2))
    (L_REP / 'g_c_dota_tile.md').write_text(
        f"# G-c DoTA-CAP tile grids: **{'PASS' if s['pass'] else 'FAIL'}**\n\n{s['ok']} / {len(report)} clips stored; "
        f"failed {len(failed)} {sorted(failed.items())[:10]}; CAP-dev missing {len(missing_dev)} {missing_dev[:10]}; "
        f"min cos {s['g_c_min_cos']}\n")
    return s


def dota_done():
    p = L_REP / 'g_c_dota_tile.json'
    return p.exists() and json.loads(p.read_text())['pass']


def nexar_job(tag, stride, ids):
    return Job(f'extract_{tag}', [[
        sys.executable, '-m', 'core.tools.v5_stage1', 'extract', '--tile', '--root', RAW, '--census', CENSUS,
        '--ids-file', write_ids(LOCAL / f'ids_{tag}.txt', ids), '--video-s1-dir', VIDEO_S1,
        '--grid-dir', L_GRID[tag], '--stride', stride, '--report', L_REP / f'extract_tile_{tag}.json',
        '--batch-size', '32', '--device', 'auto', '--out-dir', L_REP]])


def dota_job():
    import pandas as pd
    ann = pd.read_excel(CAP_RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
    type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
    alignment = json.loads(ALIGN.read_text())
    final_ids = set(json.loads(FINAL.read_text())['ids'])
    need_cap = {alignment['clips'][d]['cap_id'] for d in final_ids}
    assert len(final_ids) == 1129, len(final_ids)
    cmds = []
    for g in CAP_GROUPS:
        lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
        ids = sorted(c for c in need_cap if lo <= type_of[c] <= hi)
        parts = sorted((CAP_RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
        assert parts, f'no parts for group {g}'
        if ids:
            cmds.append([sys.executable, '-m', 'core.tools.v5_stage0', 'extract', '--tile',
                         '--alignment', ALIGN, '--final-readout', FINAL, '--parts', *parts,
                         '--work-dir', LOCAL / 'work' / g, '--v2-cache', CAP_SQUASH,
                         '--report', L_REP / f'extract_tile_dota_{g}.json', '--out-dir', LOCAL / 's0out',
                         '--grid-dir', L_GRID['dota'], '--cap-ids-file', write_ids(LOCAL / f'cap_ids_{g}.txt', ids),
                         '--batch-size', '32', '--max-batch-gb', str(constants.MMAU_STREAM_BATCH_GB),
                         '--device', 'auto'])
    return Job('extract_dota', cmds)


todo = {'train': not nexar_gate(S_TRAIN, TRAIN_IDS), 'val': not nexar_gate(S_VAL, VAL_IDS), 'dota': not dota_done()}
print('to extract:', [k for k, v in todo.items() if v] or 'nothing')
for k, need in todo.items():
    if need:                          # resume: Drive's partial copy back to VM disk, the tools skip finished videos
        print(f'{k}: restored {sync_dir(D_GRID[k], L_GRID[k], restore=True)} files from Drive')
jobs = ([nexar_job('train', S_TRAIN, TRAIN_IDS)] if todo['train'] else []) + \
       ([nexar_job('val', S_VAL, VAL_IDS)] if todo['val'] else []) + ([dota_job()] if todo['dota'] else [])
mirror = Mirror([(L_GRID[k], D_GRID[k]) for k, v in todo.items() if v] + [(L_REP, REPORTS)])
mirror.start()
t0 = time.time()
try:
    run_parallel(jobs)
finally:
    mirror.stop()
print(f'step 1 wall {(time.time() - t0) / 60:.0f} min')
if jobs:
    flush_drive()

for stride in (S_TRAIN, S_VAL):
    show(f'g1_extract_tile_s{stride}')
    assert json.loads((L_REP / f'g1_extract_tile_s{stride}.json').read_text())['pass'], f'S1-G1 s{stride} FAILED -- stop (§3.2)'
s = dota_summary() if todo['dota'] or not dota_done() else json.loads((L_REP / 'g_c_dota_tile.json').read_text())
show('g_c_dota_tile')
assert s['pass'], 'G-c on DoTA-CAP did not store every CAP-dev clip -- stop and paste g_c_dota_tile.md to Claude'
sync_dir(L_REP, REPORTS)

In [ ]:
# v5_head.npz into the train grids (CPU, seconds): needed only by training
LOCAL_DONE = set()      # caches complete on VM disk this session


def grid_local(key):
    '''The cache on VM disk: extracted this session, else one copy from Drive.'''
    if key not in LOCAL_DONE:
        n = sync_dir(D_GRID[key], L_GRID[key], restore=True)
        print(f'{key} grids on VM disk ({n} files copied from Drive)')
        LOCAL_DONE.add(key)
    return L_GRID[key]


head = D_GRID['train'] / constants.V5_HEAD_FILENAME
if not head.exists():
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'head', '--stats-dir', A2_BAKE,
         '--grid-dir', grid_local('train'), '--out-dir', L_REP])
    publish(L_GRID['train'] / constants.V5_HEAD_FILENAME, head)
    sync_dir(L_REP, REPORTS)
print(json.loads((L_REP / 'head.json').read_text()) if (L_REP / 'head.json').exists() else 'head present on Drive')

## 2. Train R1c, R3c, R6c in parallel, then J10 + S1-G4 + finite losses (GPU)

v2 Nexar A2's exact recipe (`window`, s8, 1,749 steps, `span`) + `--grid-dir` (centred 4×7×7 train cache) + the row's
`v5.*` from `constants.V5_TILE_ROWS` (R1c mean of 196 cells; R3c change top-K, no scorer; R6c tile scorer + `L_tile`
with dense negatives, λ 0.5). Trains on VM disk, publishes atomically, re-verified after one Drive flush, then
`verified.json`; a verified run is never retrained.

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
VERIFIED = 'verified.json'
L_DATA, L_A2, L_KNN = None, None, None


def run_dir(row, seed):
    return ROOT / row / f's{seed}'


def is_verified(d):
    rec, ck = d / VERIFIED, d / 'checkpoint_last.pt'
    if not (rec.exists() and ck.exists()):
        return False
    r = json.loads(rec.read_text())
    return r['step'] == STEPS and r['ckpt_bytes'] == ck.stat().st_size


def mark_verified(d):
    (LOCAL / VERIFIED).write_text(json.dumps({'step': STEPS, 'ckpt_bytes': (d / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / VERIFIED, d / VERIFIED)


def train_cmd(row, seed, out_dir):
    v5 = [x for k, v in constants.V5_TILE_ROWS[row].items() for x in ('--set', f'v5.{k}={v}')]
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={seed}',
            '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
            '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
            '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
            '--set', 'data.is_egocentric=false',
            '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}', *v5,
            '--data-dir', L_DATA, '--clip-dir', L_A2, '--knn-cache', L_KNN,
            '--grid-dir', grid_local('train'), '--output-dir', out_dir]


def train_job(row, seed):
    local = LOCAL / 'runs' / row / f's{seed}'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []

    def after():
        last = metrics_last(local / 'metrics.jsonl')
        assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row} s{seed}: J10 locally'
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            publish(local / name, run_dir(row, seed) / name)

    return Job(f'train_{row}_s{seed}', [train_cmd(row, seed, local) + resume], after)


need = [(r, s) for s in SEEDS for r in ROWS if not is_verified(run_dir(r, s))]
print('to train:', need or 'nothing (all verified on Drive)')
if need:
    L_DATA = stage(DATA, LOCAL / 'stage' / f'data_{C}')
    L_A2 = stage(A2_BAKE, LOCAL / 'stage' / 'A2_s8')
    L_KNN = stage(KNN, LOCAL / 'stage' / f'knn_{C}.npz')
    assert (grid_local('train') / constants.V5_HEAD_FILENAME).is_file(), 'no v5_head.npz in the train grids'
    t0 = time.time()
    run_parallel([train_job(r, s) for r, s in need])
    print(f'step 2 wall {(time.time() - t0) / 60:.0f} min')
    flush_drive()
    for r, s in need:
        assert ckpt_step(run_dir(r, s) / 'checkpoint_last.pt') == STEPS, f'{r} s{s}: Drive copy is not step {STEPS}'
        mark_verified(run_dir(r, s))

RUNS = [f'{r}/s{s}' for r in ROWS for s in SEEDS]
prov = L_REP / 'g4_provenance.json'
if not (prov.exists() and json.loads(prov.read_text())['pass'] and set(json.loads(prov.read_text())['runs']) == set(RUNS)):
    on_vm = all((LOCAL / 'runs' / r / f's{s}' / 'checkpoint_last.pt').exists() for r in ROWS for s in SEEDS)
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'provenance', '--root', LOCAL / 'runs' if on_vm else ROOT,
         '--a2-root', A2_RUNS, '--rows', *ROWS, '--seeds', *SEEDS, '--steps', STEPS, '--out-dir', L_REP])
show('g4_provenance')
assert json.loads(prov.read_text())['pass'], 'S1-G4 FAILED -- stop; paste g4_provenance.md to Claude'
for r in ROWS:                          # mechanics: a non-finite last-step loss stops here (read-out prints the rest)
    for s in SEEDS:
        last = metrics_last(run_dir(r, s) / 'metrics.jsonl')
        bad = {k: last[k] for k in ('mil', 'tile', 'total') if k in last and not np.isfinite(float(last[k]))}
        assert not bad, f'{r} s{s}: non-finite last-step losses {bad}'
        print(f"{r} s{s}: mil {last.get('mil'):.4f}"
              + (f", tile {last['tile']:.4f}, p abn {last['tile_p_abnormal']:.3f} / nrm {last['tile_p_normal']:.3f}, "
                 f"T {last['v5_temp']:.3f}" if 'tile' in last else ''))
sync_dir(L_REP, REPORTS)

## 3. Reads in parallel (GPU): `nexar_val`, DoTA-CAP-dev protocol B (+ time-reversed), maps dump

* `nexar_val` protocol N-B with A2's statistics and the **s3** val grids (crop starts snapped to the phase-0 row,
  §3.2) — the guardrail.
* DoTA-CAP-dev protocol B (`s1[::3]`, whole clip, native frames) on the CAP-dev A2 bake + the tile grids; the same
  with `--reverse-time` (printed control).
* Maps for R3c (untrained change map) and R6c (`max_s p`), plus the change map of every clip for the baselines.

Each read is skipped when its report already covers `SEEDS`. Inputs are staged on the main thread before any job
starts.

In [ ]:
CKPTS = {}


def ckpt(row, seed):
    '''VM-disk checkpoint: this session's training output, else one copy from Drive (with its metrics, J10).'''
    trained = LOCAL / 'runs' / row / f's{seed}' / 'checkpoint_last.pt'
    if trained.exists():
        return trained
    dst = LOCAL / 'ckpts' / row / f's{seed}'
    for name in ('metrics.jsonl', 'config.yaml', 'checkpoint_last.pt'):
        stage(run_dir(row, seed) / name, dst / name)
    return dst / 'checkpoint_last.pt'


def covers(path, key):
    return path.exists() and sorted(json.loads(path.read_text())[key]) == sorted(f's{s}' for s in SEEDS)


NEXAR_OUT, PB_OUT, PB_REV = ROOT / 'eval' / 'nexar_val', ROOT / 'eval' / 'dota' / CAP_DEV, ROOT / 'eval' / 'dota_rev' / CAP_DEV
MAPS = ROOT / 'maps'
want = {
    'nexar': [r for r in ROWS if not covers(NEXAR_OUT / r / 'nexar_readout.json', 'per_run_crop_endpoint')],
    'dota': [r for r in ROWS if not covers(PB_OUT / r / 'protocol_b_readout.json', 'per_run_macro')],
    'rev': [r for r in ROWS if not covers(PB_REV / r / 'protocol_b_readout.json', 'per_run_macro')],
    'maps': [(r, s) for r in MAP_ROWS for s in SEEDS if not (MAPS / f'maps_{r}_s{s}.npz').exists()],
}
print('to read:', {k: v for k, v in want.items() if v} or 'nothing')

jobs = []
if any(want.values()):
    runs = {r: [x for s in SEEDS for x in ('--run', f's{s}', ckpt(r, s))] for r in ROWS}
    if want['nexar']:
        L_DATA = stage(DATA, LOCAL / 'stage' / f'data_{C}')
        L_A2 = stage(A2_BAKE, LOCAL / 'stage' / 'A2_s8')
        val_x = stage(CLIP_S1, LOCAL / 'stage' / 'clip_s1_val', VAL_IDS)
        val_u = stage(VIDEO_S1, LOCAL / 'stage' / 'video_s1_val', VAL_IDS)
        val_g = grid_local('val')
    if want['dota'] or want['rev'] or want['maps']:
        d_s1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
        d_a2 = stage(DOTA_A2, LOCAL / 'stage' / 'dota_cap_dev_A2', CAP_IDS)
        d_lab = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
        d_g = grid_local('dota')
    for r in want['nexar']:
        out = LOCAL / 'nexar_val' / r
        jobs.append(Job(f'nexar_{r}', [[sys.executable, '-m', 'core.tools.nexar_eval', *runs[r], '--census', CENSUS,
                                         '--clip-s1-dir', val_x, '--video-s1-dir', val_u, '--stats-dir', L_A2,
                                         '--grid-s1-dir', val_g, '--data-dir', L_DATA,
                                         '--split', constants.V2_SPLIT_NEXAR_VAL, '--out-dir', out]],
                        lambda out=out, r=r: publish(out, NEXAR_OUT / r)))
    for key, base, extra in (('dota', PB_OUT, []), ('rev', PB_REV, ['--reverse-time'])):
        for r in want[key]:
            out = LOCAL / key / r
            jobs.append(Job(f'{key}_{r}', [[sys.executable, '-m', 'core.tools.protocol_b_eval', *runs[r],
                                             '--input-dir', d_a2, '--grid-dir', d_g, '--s1-dir', d_s1,
                                             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
                                             '--data-dir', d_lab, '--split', CAP_DEV, '--out-dir', out,
                                             '--device', 'auto', *extra]],
                            lambda out=out, r=r, base=base: publish(out, base / r)))
    for r, s in want['maps']:
        out = LOCAL / 'maps' / f'maps_{r}_s{s}.npz'
        jobs.append(Job(f'maps_{r}_s{s}', [[sys.executable, '-m', 'core.tools.v5_maps', 'dump', '--name', f'{r}_s{s}',
                                            '--ckpt', ckpt(r, s), '--input-dir', d_a2, '--grid-dir', d_g,
                                            '--s1-dir', d_s1, '--data-dir', d_lab, '--stride', S_DOTA, '--out', out]],
                        lambda out=out: (publish(out.with_suffix('.json'), MAPS / out.with_suffix('.json').name),
                                         publish(out, MAPS / out.name))))
t0 = time.time()
run_parallel(jobs)
if jobs:
    print(f'step 3 wall {(time.time() - t0) / 60:.0f} min')
    flush_drive()
for r in ROWS:
    pbr = json.loads((PB_OUT / r / 'protocol_b_readout.json').read_text())
    rev = json.loads((PB_REV / r / 'protocol_b_readout.json').read_text())
    nx = json.loads((NEXAR_OUT / r / 'nexar_readout.json').read_text())
    print(f"{r}: DoTA-CAP-dev macro {pbr['macro']['mean']:.4f} (reversed {rev['macro']['mean']:.4f}) | "
          f"nexar_val crop endpoint {nx['per_run_crop_endpoint']}")

## 4. Maps read + pilot read-out (CPU) → **paste `REPORTS/v51_pilot_readout.md` to Claude**

Maps: hit − chance, hit − centre prior, **hit − change map**, mass-in-box (− chance, − change map), by ego and
box-share bin, STAUC beside the uniform / centre / change maps (same temporal scores). Read-out: clip-macro Δ vs R1c +
TOST ±0.01, position-controlled (a) and (b), shift test, time-reversed, AP, subsets by ego / box share, `nexar_val`
guardrail, mechanics, and the §3 gate per candidate. **PRELIMINARY (n = 1)**.

In [ ]:
RAW_ANN = DOTA / 'raw_annotations'
ANN = RAW_ANN / 'annotations'
if not ANN.is_dir():
    ANN = LOCAL / 'dota_meta' / 'annotations'
    if not ANN.exists():
        ANN.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(f'cd {ANN.parent} && curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip',
                       shell=True, check=True)
META = DOTA / 'metadata_val.json'
SPLIT = write_ids(LOCAL / 'dota_meta' / 'val_ids.txt', json.loads(META.read_text()))
print('annotations:', len(list(ANN.glob('*.json'))))

maps_json = L_REP / 'maps_readout.json'
done = maps_json.exists() and set(json.loads(maps_json.read_text())['rows']) == set(MAP_ROWS) and all(
    json.loads(maps_json.read_text())['rows'][r]['seeds'] == sorted(SEEDS) for r in MAP_ROWS)
if not done:
    maps_local = {f'maps_{r}_s{s}.npz': stage(MAPS / f'maps_{r}_s{s}.npz', LOCAL / 'maps' / f'maps_{r}_s{s}.npz')
                  for r in MAP_ROWS for s in SEEDS}
    run([sys.executable, '-m', 'core.tools.v5_maps', 'read',
         *[x for r in MAP_ROWS for s in SEEDS for x in ('--run', r, s, maps_local[f'maps_{r}_s{s}.npz'])],
         '--stride', S_DOTA, '--metadata', META, '--split-file', SPLIT, '--annotations-dir', ANN, '--out-dir', L_REP])

run([sys.executable, '-m', 'core.tools.v5_tile_read', '--pb-dir', PB_OUT, '--pb-rev-dir', PB_REV,
     '--nexar-dir', NEXAR_OUT, '--maps-readout', maps_json, '--rows', *ROWS, '--control', CONTROL,
     '--seeds', *SEEDS, '--stride', S_DOTA, '--runs-root', ROOT, '--annotations-dir', ANN,
     '--metadata', META, '--split-file', SPLIT, '--out-dir', L_REP])
print((L_REP / 'v51_pilot_readout.md').read_text())
sync_dir(L_REP, REPORTS)
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/v51_pilot_readout.md (+ maps_readout.md) to Claude')